In [8]:
import json

input_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\generated-responses_Qwen3-32B_partaa_with-leakage-spans.jsonl"
output_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\generated-responses_Qwen3-32B_with-leakage-spans.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        if not line.strip():
            continue

        data = json.loads(line)

        # Remove leakage_spans if it is empty
        if data.get("leakage_spans") == []:
            continue

        fout.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Saved cleaned file to: {output_path}")

Saved cleaned file to: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\generated-responses_Qwen3-32B_with-leakage-spans.jsonl


In [4]:
from pathlib import Path
import json
import random
import shutil


src = Path(
    r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-32B\generated-responses_Qwen3-32B_with-leakage-spans.jsonl"
)

backup = src.with_suffix(src.suffix + ".bak")
test_path = src.with_name("test_" + src.name)
val_path = src.with_name("val_" + src.name)


# Create backup if it does not already exist
if not backup.exists():
    shutil.copy2(src, backup)


# Always split from the original backup so rerunning is safe
records = [
    json.loads(line)
    for line in backup.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Deterministic shuffle
random.Random(42).shuffle(records)


# Calculate split sizes
n = len(records)
n_test = round(n * 0.15)
n_val = round(n * 0.15)


# Create splits: 15% test, 15% validation, 70% train
splits = {
    test_path: records[:n_test],
    val_path: records[n_test:n_test + n_val],
    src: records[n_test + n_val:],
}


# Write each split
for path, rows in splits.items():
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# Print summary
print(f"total={n}")
print(f"test={len(splits[test_path])}: {test_path}")
print(f"val={len(splits[val_path])}: {val_path}")
print(f"train={len(splits[src])}: {src}")
print(f"backup={backup}")

total=892
test=134: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-32B\test_generated-responses_Qwen3-32B_with-leakage-spans.jsonl
val=134: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-32B\val_generated-responses_Qwen3-32B_with-leakage-spans.jsonl
train=624: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-32B\generated-responses_Qwen3-32B_with-leakage-spans.jsonl
backup=H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-32B\generated-responses_Qwen3-32B_with-leakage-spans.jsonl.bak
